### 1. word2vec


#### 1-1 실습


In [1]:
import torch
import torch.nn as nn

# 설정
vocab_size = 5  # 어휘 사전 크기 (V)
embedding_dim = 3  # 투영층 차원 (M)

# 1. 동일한 가중치 행렬(W) 공유를 위해 시드 고정
torch.manual_seed(42)

# 방식 A: nn.Embedding 사용 (Lookup Table 방식)
embedding_layer = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embedding_dim)

# 방식 B: nn.Linear 사용 (행렬 곱 방식, 편향 bias=False)
linear_layer = nn.Linear(in_features=vocab_size, out_features=embedding_dim, bias=False)

# 두 레이어의 가중치 행렬 W를 동일하게 맞춤 (Embedding 가중치를 Linear에 전치하여 복사)
linear_layer.weight.data = embedding_layer.weight.data.T

# -------------------------------------------------------------
# 테스트: 1번 단어("cat")의 투영층 벡터 추출
target_word_idx = 1  # 1번 단어 인덱스

# A. nn.Embedding 방식
# 정수 인덱스 텐서를 입력으로 바로 전달
input_idx = torch.tensor([target_word_idx])
embed_output = embedding_layer(input_idx)

# B. nn.Linear 방식
# 원-핫 벡터 [0, 1, 0, 0, 0] 생성 후 행렬 곱 수행
one_hot_input = torch.zeros(1, vocab_size)
one_hot_input[0, target_word_idx] = 1.0
linear_output = linear_layer(one_hot_input)

# -------------------------------------------------------------
print("--- [결과 비교] ---")
print("nn.Embedding 출력 (Lookup) :\n", embed_output.detach().numpy())
print("nn.Linear    출력 (MatMul) :\n", linear_output.detach().numpy())
print("\n두 결과가 완벽히 동일한가?:", torch.allclose(embed_output, linear_output))

--- [결과 비교] ---
nn.Embedding 출력 (Lookup) :
 [[ 0.23033303 -1.1228564  -0.18632829]]
nn.Linear    출력 (MatMul) :
 [[ 0.23033303 -1.1228564  -0.18632829]]

두 결과가 완벽히 동일한가?: True


In [2]:
import torch
import torch.nn as nn

# -------------------------------------------------------------
# 1. 설정 (이미지의 실제값 조건 적용)
# -------------------------------------------------------------
vocab_size = 3  # V = 3 (0: 사과, 1: 바나나, 2: 체리)
embedding_dim = 2  # M = 2 (2차원 공간)
target_idx = 1  # 선택할 단어: 1번 "바나나"

# 이미님의 설정과 동일한 가중치 행렬 W (3 x 2) 정의
# W = [[ 0.5,  0.8],
#      [-0.3,  0.9],
#      [ 0.1, -0.4]]
weight_matrix = torch.tensor(
    [[0.5, 0.8], [-0.3, 0.9], [0.1, -0.4]], dtype=torch.float32
)

# -------------------------------------------------------------
# 2. nn.Embedding 구현 (Lookup 연산 방식)
# -------------------------------------------------------------
embedding_layer = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embedding_dim)

# 동일한 가중치를 사용하도록 가중치 수동 할당
embedding_layer.weight.data = weight_matrix.clone()

# 입력: 정수 인덱스 텐서 (LongTensor)
input_idx = torch.tensor([target_idx], dtype=torch.long)

# Lookup 실행
output_embed = embedding_layer(input_idx)

# -------------------------------------------------------------
# 3. nn.Linear 구현 (원-핫 벡터 행렬 곱 연산 방식)
# -------------------------------------------------------------
# bias=False 설정하여 순수 행렬 곱만 수행
linear_layer = nn.Linear(in_features=vocab_size, out_features=embedding_dim, bias=False)

# nn.Linear의 가중치는 (out_features, in_features) 형태이므로 전치(Transpose)하여 복사
linear_layer.weight.data = weight_matrix.T.clone()

# 입력: 1번 단어("바나나")에 대한 원-핫 벡터 생성 [0, 1, 0] (FloatTensor)
one_hot_input = torch.tensor([[0.0, 1.0, 0.0]], dtype=torch.float32)

# 행렬 곱 실행
output_linear = linear_layer(one_hot_input)

# -------------------------------------------------------------
# 4. 결과 출력 및 검증
# -------------------------------------------------------------
print("--- [출력 결과] ---")
print("1. nn.Embedding (Lookup) 결과 :", output_embed.detach().numpy())
print("2. nn.Linear    (MatMul) 결과 :", output_linear.detach().numpy())

# 두 결과가 수학적으로 일치하는지 확인
print("\n두 연산 결과가 완벽히 동일한가?:", torch.allclose(output_embed, output_linear))

--- [출력 결과] ---
1. nn.Embedding (Lookup) 결과 : [[-0.3  0.9]]
2. nn.Linear    (MatMul) 결과 : [[-0.3  0.9]]

두 연산 결과가 완벽히 동일한가?: True


#### 1-2 CBOW 모델에서의 nn.Embedding 실제 연산 과정


In [3]:
import torch
import torch.nn as nn


class CBOWProjection(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super(CBOWProjection, self).__init__()
        # 1. Lookup Table 역할을 하는 임베딩 레이어 (Projection Layer)
        self.embeddings = nn.Embedding(vocab_size, embedding_dim)

    def forward(self, context_indices):
        # context_indices 형태: (batch_size, num_context_words)
        # 예: [[0, 2, 3, 4]] -> 주변 단어 4개의 인덱스

        # 2. Lookup 연산으로 주변 단어 벡터들을 한 번에 추출
        # (batch_size, num_context_words, embedding_dim)
        context_embeds = self.embeddings(context_indices)

        # 3. 투영층 핵심 연산: 주변 단어 벡터들의 평균(Mean) 계산
        # (batch_size, embedding_dim)
        projection_vector = torch.mean(context_embeds, dim=0)
        return projection_vector


# 실행 예시
cbow_proj = CBOWProjection(vocab_size=10000, embedding_dim=300)

# 배치 크기 1, 주변 단어 4개 (인덱스: 12, 45, 99, 301)
context = torch.tensor([12, 45, 99, 301])

output_vector = cbow_proj(context)
print("CBOW 투영층 최종 평균 벡터 크기:", output_vector.shape)  # [1, 300]

CBOW 투영층 최종 평균 벡터 크기: torch.Size([300])


#### 1-3 CBOW 구현 (Context $\rightarrow$ Target)


In [4]:
import torch
import torch.nn as nn

# 사전 정의
word2idx = {"the": 0, "cat": 1, "sat": 2, "on": 3, "mat": 4}
vocab_size = len(word2idx)
embedding_dim = 10  # M = 10차원 공간

# 입력 데이터 설정 ('the', 'cat', 'on', 'the') 및 정답 Target ('sat')
context_indices = [0, 1, 3, 0]
target_index = 2  # 중심 단어: 'sat'


# -------------------------------------------------------------
# 1. nn.Embedding 기반 CBOW (Lookup 방식)
# -------------------------------------------------------------
class CBOWEmbeddingModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super(CBOWEmbeddingModel, self).__init__()
        # W 행렬 (Input-to-Hidden): Lookup Table
        self.embeddings = nn.Embedding(vocab_size, embedding_dim)
        # W' 행렬 (Hidden-to-Output)
        self.linear = nn.Linear(embedding_dim, vocab_size, bias=False)

    def forward(self, inputs):
        # inputs: [0, 1, 3, 0] (LongTensor)
        embeds = self.embeddings(inputs)  # (4, 10)
        # 4개 주변 단어 임베딩의 평균 -> (1, 10) 투영층
        context_vector = torch.mean(embeds, dim=0, keepdim=True)
        # W'를 곱해 5개 단어에 대한 Logits 생성
        out = self.linear(context_vector)  # (1, 5)
        return out


cbow_embed = CBOWEmbeddingModel(vocab_size, embedding_dim)
context_idx = torch.tensor(context_indices, dtype=torch.long)
output_embed = cbow_embed(context_idx)


# -------------------------------------------------------------
# 2. nn.Linear 기반 CBOW (One-Hot MatMul 방식)
# -------------------------------------------------------------
class CBOWLinearModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super(CBOWLinearModel, self).__init__()
        # W 행렬 (Input-to-Hidden): 행렬 곱
        self.linear1 = nn.Linear(vocab_size, embedding_dim, bias=False)
        # W' 행렬 (Hidden-to-Output)
        self.linear2 = nn.Linear(embedding_dim, vocab_size, bias=False)

    def forward(self, one_hot_inputs):
        # one_hot_inputs: (4, 5) 원-핫 행렬 (FloatTensor)
        embeds = self.linear1(one_hot_inputs)  # (4, 10) 행렬 곱
        # 4개 주변 단어 임베딩의 평균 -> (1, 10) 투영층
        context_vector = torch.mean(embeds, dim=0, keepdim=True)
        # W'를 곱해 5개 단어에 대한 Logits 생성
        out = self.linear2(context_vector)  # (1, 5)
        return out


cbow_linear = CBOWLinearModel(vocab_size, embedding_dim)

# 두 모델의 가중치(W, W')를 완벽히 동일하게 맞춰 결과 비교
cbow_linear.linear1.weight.data = cbow_embed.embeddings.weight.data.T.clone()
cbow_linear.linear2.weight.data = cbow_embed.linear.weight.data.clone()

# 입력: ['the', 'cat', 'on', 'the'] 4개 단어에 대한 원-핫 벡터 생성 (4, 5)
one_hot_context = torch.zeros(len(context_indices), vocab_size, dtype=torch.float32)
for i, idx in enumerate(context_indices):
    one_hot_context[i][idx] = 1.0

output_linear = cbow_linear(one_hot_context)

# -------------------------------------------------------------
# 3. 결과 비교 및 손실(Loss) 계산
# -------------------------------------------------------------
print("--- [CBOW Logits 출력 크기] ---")
print("1. nn.Embedding 기반 CBOW 출력 크기 :", output_embed.shape)  # torch.Size([1, 5])
print(
    "2. nn.Linear    기반 CBOW 출력 크기 :", output_linear.shape
)  # torch.Size([1, 5])

print("\n--- [연산 결과 동일 여부 확인] ---")
print(
    "두 모델의 Logits 연산 결과가 일치하는가?:",
    torch.allclose(output_embed, output_linear),
)

# 손실 함수 및 정답 Target 설정
criterion = nn.CrossEntropyLoss()
target_tensor = torch.tensor([target_index], dtype=torch.long)  # 'sat' (인덱스 2)

# 두 모델의 예측 점수(Logits)와 정답 Target 비교
loss_embed = criterion(output_embed, target_tensor)
loss_linear = criterion(output_linear, target_tensor)

print("\n--- [CBOW 손실(Loss) 계산 결과] ---")
print("1. nn.Embedding 기반 CBOW Loss :", loss_embed.item())
print("2. nn.Linear    기반 CBOW Loss :", loss_linear.item())
print("두 손실값이 완벽히 동일한가?   :", torch.allclose(loss_embed, loss_linear))

--- [CBOW Logits 출력 크기] ---
1. nn.Embedding 기반 CBOW 출력 크기 : torch.Size([1, 5])
2. nn.Linear    기반 CBOW 출력 크기 : torch.Size([1, 5])

--- [연산 결과 동일 여부 확인] ---
두 모델의 Logits 연산 결과가 일치하는가?: True

--- [CBOW 손실(Loss) 계산 결과] ---
1. nn.Embedding 기반 CBOW Loss : 1.8189280033111572
2. nn.Linear    기반 CBOW Loss : 1.8189280033111572
두 손실값이 완벽히 동일한가?   : True


#### 1-4 Skip-gram 구현 (Target $\rightarrow$ Context)


In [5]:
import torch
import torch.nn as nn

# 사전 정의
word2idx = {"the": 0, "cat": 1, "sat": 2, "on": 3, "mat": 4}
vocab_size = len(word2idx)
embedding_dim = 10  # M = 10차원 공간

# 데이터 설정 (중심 단어: 'sat'(2) / 주변 단어들: ['the'(0), 'cat'(1), 'on'(3), 'the'(0)])
target_index = 2
context_indices = [0, 1, 3, 0]


# -------------------------------------------------------------
# 1. nn.Embedding 기반 Skip-gram (Lookup 방식)
# -------------------------------------------------------------
class SkipGramEmbeddingModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super(SkipGramEmbeddingModel, self).__init__()
        # W 행렬 (Input-to-Hidden): Lookup Table
        self.embeddings = nn.Embedding(vocab_size, embedding_dim)
        # W' 행렬 (Hidden-to-Output)
        self.linear = nn.Linear(embedding_dim, vocab_size, bias=False)

    def forward(self, target_input):
        # target_input: [2] ('sat' 정수 인덱스)
        embed = self.embeddings(target_input)  # (1, 10) 투영층
        # W'를 곱해 5개 단어에 대한 예측 점수(Logits) 생성
        out = self.linear(embed)  # (1, 5)
        return out


sg_embed_model = SkipGramEmbeddingModel(vocab_size, embedding_dim)


# -------------------------------------------------------------
# 2. nn.Linear 기반 Skip-gram (One-Hot MatMul 방식)
# -------------------------------------------------------------
class SkipGramLinearModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super(SkipGramLinearModel, self).__init__()
        # W 행렬 (Input-to-Hidden): 행렬 곱
        self.linear1 = nn.Linear(vocab_size, embedding_dim, bias=False)
        # W' 행렬 (Hidden-to-Output)
        self.linear2 = nn.Linear(embedding_dim, vocab_size, bias=False)

    def forward(self, target_one_hot):
        # target_one_hot: [[0.0, 0.0, 1.0, 0.0, 0.0]] (원-핫 벡터)
        embed = self.linear1(target_one_hot)  # (1, 10) 행렬 곱
        # W'를 곱해 5개 단어에 대한 예측 점수(Logits) 생성
        out = self.linear2(embed)  # (1, 5)
        return out


sg_linear_model = SkipGramLinearModel(vocab_size, embedding_dim)

# 두 모델의 가중치(W, W')를 완벽히 동일하게 맞춰 연산 일치 확인
sg_linear_model.linear1.weight.data = sg_embed_model.embeddings.weight.data.T.clone()
sg_linear_model.linear2.weight.data = sg_embed_model.linear.weight.data.clone()

# -------------------------------------------------------------
# 3. 입력 생성 및 예측 출력 계산
# -------------------------------------------------------------
# 1) Embedding용 입력: LongTensor 인덱스 [2]
target_idx_tensor = torch.tensor([target_index], dtype=torch.long)
output_embed = sg_embed_model(target_idx_tensor)

# 2) Linear용 입력: FloatTensor 원-핫 벡터 [0, 0, 1, 0, 0]
target_one_hot_tensor = torch.zeros(1, vocab_size, dtype=torch.float32)
target_one_hot_tensor[0][target_index] = 1.0
output_linear = sg_linear_model(target_one_hot_tensor)

# -------------------------------------------------------------
# 4. 손실(Loss) 계산 및 검증
# -------------------------------------------------------------
criterion = nn.CrossEntropyLoss()
context_targets = torch.tensor(
    context_indices, dtype=torch.long
)  # 정답 주변 단어들 [0, 1, 3, 0]

# 1) Embedding 모델 손실 계산
loss_embed = 0
for context in context_targets:
    loss_embed += criterion(output_embed, context.unsqueeze(0))
    print("loss_embed : ", loss_embed)

# 2) Linear 모델 손실 계산
loss_linear = 0
for context in context_targets:
    loss_linear += criterion(output_linear, context.unsqueeze(0))

print("--- [Skip-gram 출력 크기] ---")
print("1. nn.Embedding Logits 출력 크기 :", output_embed.shape)  # torch.Size([1, 5])
print("2. nn.Linear xLogits 출력 크기 :", output_linear.shape)  # torch.Size([1, 5])
print(
    "3. 두 모델의 Logits 예측 값이 완전히 일치하는가? :",
    torch.allclose(output_embed, output_linear),
)

print("\n--- [Logits 및 손실(Loss) 수치 일치 확인] ---")
print("1. nn.Embedding 총 손실(Loss)                     :", loss_embed.item())
for i, context in enumerate(context_targets):
    print(f"nn.Embedding 총 손실[{i}] : {context}")

print("2. nn.Linear 총 손실(Loss) :", loss_linear.item())
print(
    "3. 두 모델의 손실(Loss)이 완전히 일치하는가? :",
    torch.allclose(loss_embed, loss_linear),
)

loss_embed :  tensor(1.7967, grad_fn=<AddBackward0>)
loss_embed :  tensor(4.7623, grad_fn=<AddBackward0>)
loss_embed :  tensor(6.5753, grad_fn=<AddBackward0>)
loss_embed :  tensor(8.3719, grad_fn=<AddBackward0>)
--- [Skip-gram 출력 크기] ---
1. nn.Embedding Logits 출력 크기 : torch.Size([1, 5])
2. nn.Linear xLogits 출력 크기 : torch.Size([1, 5])
3. 두 모델의 Logits 예측 값이 완전히 일치하는가? : True

--- [Logits 및 손실(Loss) 수치 일치 확인] ---
1. nn.Embedding 총 손실(Loss)                     : 8.371932983398438
nn.Embedding 총 손실[0] : 0
nn.Embedding 총 손실[1] : 1
nn.Embedding 총 손실[2] : 3
nn.Embedding 총 손실[3] : 0
2. nn.Linear 총 손실(Loss) : 8.371932983398438
3. 두 모델의 손실(Loss)이 완전히 일치하는가? : True


#### 1-5 CBOW 구현 코드 (nn.Embedding vs nn.Linear)


In [6]:
import torch
import torch.nn as nn

# -------------------------------------------------------------
# 1. 환경 및 동일 가중치 행렬 W 설정
# -------------------------------------------------------------
vocab_size = 3  # V = 3 (0: 사과, 1: 바나나, 2: 체리)
embedding_dim = 2  # M = 2
context_indices = [0, 2]  # 주변 단어: 사과(0), 체리(2)
target_index = 1  # 정답 중심 단어: 바나나(1)

# 가중치 행렬 W (Input-to-Hidden)
weight_matrix = torch.tensor(
    [
        [0.5, 0.8],  # 0번 사과
        [-0.3, 0.9],  # 1번 바나나
        [0.1, -0.4],  # 2번 체리
    ],
    dtype=torch.float32,
)


# -------------------------------------------------------------
# 2. nn.Embedding 기반 CBOW (Lookup + Mean)
# -------------------------------------------------------------
class CBOWEmbedding(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super(CBOWEmbedding, self).__init__()
        self.embeddings = nn.Embedding(vocab_size, embedding_dim)
        self.embeddings.weight.data = weight_matrix.clone()

        # W' 행렬 (Hidden-to-Output): 투영층(2차원) -> 출력층(3차원 Vocab)
        self.output_layer = nn.Linear(embedding_dim, vocab_size, bias=False)

    def forward(self, inputs):
        # inputs: [0, 2] 형태의 LongTensor
        embeds = self.embeddings(inputs)  # (2, 2) 크기의 임베딩 벡터 추출
        # 주변 단어 임베딩의 평균(Mean) 계산 -> 투영층(Projection Layer) 출력 (1, 2)
        context_vector = torch.mean(embeds, dim=0, keepdim=True)
        # W'를 곱해 3개 단어에 대한 예측 점수(Logits) 생성 -> (1, 3)
        logits = self.output_layer(context_vector)
        return context_vector, logits


cbow_embed_model = CBOWEmbedding(vocab_size, embedding_dim)
context_tensor_idx = torch.tensor(context_indices, dtype=torch.long)
output_embed, logits_embed = cbow_embed_model(context_tensor_idx)


# -------------------------------------------------------------
# 3. nn.Linear 기반 CBOW (One-Hot MatMul + Mean)
# -------------------------------------------------------------
class CBOWLinear(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super(CBOWLinear, self).__init__()
        self.linear = nn.Linear(vocab_size, embedding_dim, bias=False)
        self.linear.weight.data = weight_matrix.T.clone()

        # W' 행렬 (Hidden-to-Output): 동일한 가중치 사용
        self.output_layer = nn.Linear(embedding_dim, vocab_size, bias=False)

    def forward(self, inputs):
        # inputs: 원-핫 벡터들의 배치 (2, 3)
        embeds = self.linear(inputs)  # (2, 2) 행렬 곱 연산
        # 주변 단어 임베딩의 평균(Mean) 계산 (1, 2)
        context_vector = torch.mean(embeds, dim=0, keepdim=True)
        # W'를 곱해 예측 점수(Logits) 생성 -> (1, 3)
        logits = self.output_layer(context_vector)
        return context_vector, logits


cbow_linear_model = CBOWLinear(vocab_size, embedding_dim)

# 두 모델의 W' 가중치를 동일하게 맞춤
cbow_linear_model.output_layer.weight.data = (
    cbow_embed_model.output_layer.weight.data.clone()
)

# 사과(0번)와 체리(2번)의 원-핫 벡터
context_one_hots = torch.tensor(
    [
        [1.0, 0.0, 0.0],  # 0번 사과
        [0.0, 0.0, 1.0],  # 2번 체리
    ],
    dtype=torch.float32,
)

output_linear, logits_linear = cbow_linear_model(context_one_hots)

# -------------------------------------------------------------
# 4. 결과 검증 및 손실(Loss) 계산
# -------------------------------------------------------------
print("--- [CBOW 투영층(Projection) 출력 결과] ---")
print("1. nn.Embedding CBOW 결과 :", output_embed.detach().numpy())
print("2. nn.Linear    CBOW 결과 :", output_linear.detach().numpy())

print("\n두 결과가 동일한가?:", torch.allclose(output_embed, output_linear))

# 손실(Loss) 계산
criterion = nn.CrossEntropyLoss()

# 정답 Target: 중심 단어 1번 '바나나'
target_tensor = torch.tensor([target_index], dtype=torch.long)

# 1 x 3 차원의 Logits와 정답 Target([1])을 비교하여 Loss 산출
loss_embed = criterion(logits_embed, target_tensor)
loss_linear = criterion(logits_linear, target_tensor)

print("\n--- [CBOW 손실(Loss) 계산 결과] ---")
print("1. nn.Embedding 기반 CBOW Loss :", loss_embed.item())
print("2. nn.Linear    기반 CBOW Loss :", loss_linear.item())
print("두 손실값이 완벽히 동일한가?   :", torch.allclose(loss_embed, loss_linear))

--- [CBOW 투영층(Projection) 출력 결과] ---
1. nn.Embedding CBOW 결과 : [[0.3 0.2]]
2. nn.Linear    CBOW 결과 : [[0.3 0.2]]

두 결과가 동일한가?: True

--- [CBOW 손실(Loss) 계산 결과] ---
1. nn.Embedding 기반 CBOW Loss : 1.0679725408554077
2. nn.Linear    기반 CBOW Loss : 1.0679725408554077
두 손실값이 완벽히 동일한가?   : True


#### 1-6 Skip-gram구현 코드 (nn.Embedding vs nn.Linear)


In [7]:
import torch
import torch.nn as nn

# -------------------------------------------------------------
# 1. 환경 및 동일 가중치 행렬 W 설정
# -------------------------------------------------------------
vocab_size = 3  # V = 3 (0: 사과, 1: 바나나, 2: 체리)
embedding_dim = 2  # M = 2

# 가중치 행렬 W (3 x 2)
weight_matrix = torch.tensor(
    [
        [0.5, 0.8],  # 0번 사과
        [-0.3, 0.9],  # 1번 바나나 (Target)
        [0.1, -0.4],  # 2번 체리
    ],
    dtype=torch.float32,
)


# -------------------------------------------------------------
# 2. nn.Embedding 기반 Skip-gram (Lookup 방식)
# -------------------------------------------------------------
class SkipGramEmbedding(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super(SkipGramEmbedding, self).__init__()
        # W 행렬 (Input-to-Hidden)
        self.embeddings = nn.Embedding(vocab_size, embedding_dim)
        self.embeddings.weight.data = weight_matrix.clone()

        # W' 행렬 (Hidden-to-Output)
        self.linear = nn.Linear(embedding_dim, vocab_size, bias=False)

    def forward(self, target_input):
        # target_input: [1] (바나나 인덱스)
        # 1. W 행렬에서 '바나나'의 2차원 임베딩 추출 -> 투영층 Output
        embed = self.embeddings(target_input)  # (1, 2)

        # 2. W' 행렬을 곱해 어휘 사전(3개)에 대한 예측 Score(Logits) 출력
        logits = self.linear(embed)  # (1, 3)
        return logits


target_tensor_idx = torch.tensor([1], dtype=torch.long)  # Target: 1번 '바나나'
sg_embed_model = SkipGramEmbedding(vocab_size, embedding_dim)
output_embed = sg_embed_model(target_tensor_idx)


# -------------------------------------------------------------
# 3. nn.Linear 기반 Skip-gram (One-Hot MatMul 방식)
# -------------------------------------------------------------
class SkipGramLinear(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super(SkipGramLinear, self).__init__()
        # W 행렬
        self.linear1 = nn.Linear(vocab_size, embedding_dim, bias=False)
        self.linear1.weight.data = weight_matrix.T.clone()

        # W' 행렬
        self.linear2 = nn.Linear(embedding_dim, vocab_size, bias=False)

    def forward(self, target_one_hot):
        # target_one_hot: [[0.0, 1.0, 0.0]] (바나나 원-핫)
        # 1. W 행렬과 원-핫 행렬 곱 -> 투영층 Output
        embed = self.linear1(target_one_hot)  # (1, 2)

        # 2. W' 행렬을 곱해 Logits 출력
        logits = self.linear2(embed)  # (1, 3)
        return logits


target_one_hot = torch.tensor(
    [[0.0, 1.0, 0.0]], dtype=torch.float32
)  # Target: '바나나' 원-핫
sg_linear_model = SkipGramLinear(vocab_size, embedding_dim)

# 두 모델의 W' 가중치를 동일하게 맞춤
sg_linear_model.linear2.weight.data = sg_embed_model.linear.weight.data.clone()

output_linear = sg_linear_model(target_one_hot)

# -------------------------------------------------------------
# 4. 결과 검증 및 손실(Loss) 계산
# -------------------------------------------------------------
print("--- [Skip-gram 출력 결과 (Logits)] ---")
print("1. nn.Embedding Skip-gram 결과 :", output_embed.detach().numpy())
print("2. nn.Linear    Skip-gram 결과 :", output_linear.detach().numpy())
print("\n두 결과가 동일한가?:", torch.allclose(output_embed, output_linear))

# 손실(Loss) 계산 예시: 정답인 주변 단어 0번('사과')과 2번('체리') 각각에 대해 계산
criterion = nn.CrossEntropyLoss()
context_targets = torch.tensor([0, 2], dtype=torch.long)  # 주변 단어들

loss = 0
for context in context_targets:
    loss += criterion(output_embed, context.unsqueeze(0))

print("\n주변 단어 [0, 2]에 대한 총 손실(Loss):", loss.item())

--- [Skip-gram 출력 결과 (Logits)] ---
1. nn.Embedding Skip-gram 결과 : [[-0.15000992 -0.35325602  0.57683104]]
2. nn.Linear    Skip-gram 결과 : [[-0.15000992 -0.35325602  0.57683104]]

두 결과가 동일한가?: True

주변 단어 [0, 2]에 대한 총 손실(Loss): 1.9872057437896729


#### 1-7 영화 리뷰 데이터셋(daum_movie_review.csv) + CBOW (Continuous Bag-of-Words)


In [9]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import re
import pickle
from collections import Counter
from sklearn.model_selection import train_test_split

# -------------------------------------------------------------
# 1. 데이터 로드, 전처리 및 사전 구축
# -------------------------------------------------------------
df = pd.read_csv("../data/daum_movie_review.csv")
df.head()
reviews = df["review"].dropna().tolist()


def clean_text(text):
    text = re.sub(r"[^가-힣a-zA-Z0-9\s]", "", str(text))
    return text.strip()


cleaned_reviews = [clean_text(r) for r in reviews if len(clean_text(r)) > 0]
tokens_list = [review.split() for review in cleaned_reviews]

# 단어 사전 구축 (최소 빈도수 5 이상)
MIN_COUNT = 5
word_counts = Counter([word for tokens in tokens_list for word in tokens])
vocab = [word for word, count in word_counts.items() if count >= MIN_COUNT]

word2idx = {word: i for i, word in enumerate(vocab)}
idx2word = {i: word for i, word in enumerate(vocab)}
vocab_size = len(vocab)

# -------------------------------------------------------------
# 2. CBOW 타겟-문맥 쌍 생성 및 Train/Val/Test 분할
# -------------------------------------------------------------
WINDOW_SIZE = 2
cbow_pairs = []

for tokens in tokens_list:
    indices = [word2idx[word] for word in tokens if word in word2idx]
    if len(indices) < WINDOW_SIZE * 2 + 1:
        continue
    for i in range(WINDOW_SIZE, len(indices) - WINDOW_SIZE):
        context = indices[i - WINDOW_SIZE : i] + indices[i + 1 : i + WINDOW_SIZE + 1]
        target = indices[i]
        cbow_pairs.append((context, target))

# 80% Train, 10% Validation, 10% Test 비율 분할
train_pairs, test_pairs = train_test_split(cbow_pairs, test_size=0.2, random_state=42)
val_pairs, test_pairs = train_test_split(test_pairs, test_size=0.5, random_state=42)

print(f"전체 샘플 수    : {len(cbow_pairs)}")
print(f"Train 샘플 수   : {len(train_pairs)} (80%)")
print(f"Val 샘플 수     : {len(val_pairs)} (10%)")
print(f"Test 샘플 수    : {len(test_pairs)} (10%)")


# Dataset 클래스 (미리 tensor로 변환하도록 성능 최적화)
class CBOWDataset(Dataset):
    def __init__(self, pairs):
        contexts = [p[0] for p in pairs]
        targets = [p[1] for p in pairs]
        self.contexts = torch.tensor(contexts, dtype=torch.long)
        self.targets = torch.tensor(targets, dtype=torch.long)

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, idx):
        return self.contexts[idx], self.targets[idx]


train_loader = DataLoader(CBOWDataset(train_pairs), batch_size=512, shuffle=True)
val_loader = DataLoader(CBOWDataset(val_pairs), batch_size=512, shuffle=False)


# -------------------------------------------------------------
# 3. CBOW 모델 정의
# -------------------------------------------------------------
class CBOWModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super(CBOWModel, self).__init__()
        self.embeddings = nn.Embedding(vocab_size, embedding_dim)
        self.linear = nn.Linear(embedding_dim, vocab_size, bias=False)

    def forward(self, inputs):
        embeds = self.embeddings(inputs)
        context_vector = torch.mean(embeds, dim=1)
        out = self.linear(context_vector)
        return out


EMBEDDING_DIM = 50
model = CBOWModel(vocab_size, EMBEDDING_DIM)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.005)

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model.to(device)

# -------------------------------------------------------------
# 4. Train 및 Validation 학습 (Best Model 저장)
# -------------------------------------------------------------
epochs = 10
best_val_loss = float("inf")

print("\n--- CBOW 모델 학습 (Train & Validation) ---")
for epoch in range(epochs):
    # Train Phase
    model.train()
    train_loss = 0
    for context, target in train_loader:
        context, target = context.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(context)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    avg_train_loss = train_loss / len(train_loader)

    # Validation Phase
    model.eval()
    val_loss = 0
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for context, target in val_loader:
            context, target = context.to(device), target.to(device)
            output = model(context)
            loss = criterion(output, target)
            val_loss += loss.item()

            preds = torch.argmax(output, dim=1)
            val_correct += (preds == target).sum().item()
            val_total += target.size(0)

    avg_val_loss = val_loss / len(val_loader)
    val_acc = (val_correct / val_total) * 100

    print(
        f"Epoch {epoch + 1:02d}/{epochs} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.2f}%"
    )

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), "best_cbow_model.pth")

# 통일된 파일명으로 메타 정보 저장
test_meta = {
    "word2idx": word2idx,
    "idx2word": idx2word,
    "vocab_size": vocab_size,
    "embedding_dim": EMBEDDING_DIM,
    "test_pairs": test_pairs,
}
with open("cbow_test_meta.pkl", "wb") as f:
    pickle.dump(test_meta, f)

print("\n[저장 완료] 'best_cbow_model.pth' 및 메타데이터 저장됨.")

# -------------------------------------------------------------
# 5. 저장된 모델 로드 후 Test 데이터셋 평가
# -------------------------------------------------------------
print("\n--- [Test Phase] 저장된 모델 로드 및 평가 ---")

# 1) 통일된 파일명으로 메타 정보 불러오기
with open("cbow_test_meta.pkl", "rb") as f:
    loaded_meta = pickle.load(f)

# 2) 모델 구조 재구성 및 저장된 가중치(.pth) 로드
test_model = CBOWModel(loaded_meta["vocab_size"], loaded_meta["embedding_dim"])
test_model.load_state_dict(torch.load("best_cbow_model.pth"))
test_model.to(device)
test_model.eval()

# 3) Test DataLoader 생성 및 평가 진행
loaded_test_loader = DataLoader(
    CBOWDataset(loaded_meta["test_pairs"]), batch_size=512, shuffle=False
)

test_loss = 0
test_correct = 0
test_total = 0

with torch.no_grad():
    for context, target in loaded_test_loader:
        context, target = context.to(device), target.to(device)
        output = test_model(context)
        loss = criterion(output, target)
        test_loss += loss.item()

        preds = torch.argmax(output, dim=1)
        test_correct += (preds == target).sum().item()
        test_total += target.size(0)

avg_test_loss = test_loss / len(loaded_test_loader)
test_acc = (test_correct / test_total) * 100

print(f"Test Dataset Loss : {avg_test_loss:.4f}")
print(f"Test Accuracy     : {test_acc:.2f}% ({test_correct}/{test_total} 개 적중)")


# -------------------------------------------------------------
# 6. 저장된 모델 기준 코사인 유사도(Cosine Similarity) 평가
# -------------------------------------------------------------
def get_similar_words(word, model, meta, top_n=5):
    w2i = meta["word2idx"]
    i2w = meta["idx2word"]

    if word not in w2i:
        return f"'{word}' 단어가 어휘 사전에 없습니다."

    word_idx = torch.tensor([w2i[word]]).to(device)
    embed_weights = model.embeddings.weight.data

    target_vec = embed_weights[word_idx]

    # 코사인 유사도 계산
    norm_target = target_vec / torch.norm(target_vec, dim=1, keepdim=True)
    norm_weights = embed_weights / torch.norm(embed_weights, dim=1, keepdim=True)

    cosine_sim = torch.mm(norm_target, norm_weights.T).squeeze(0)
    top_indices = torch.topk(cosine_sim, top_n + 1).indices.tolist()

    return [
        (i2w[idx], round(cosine_sim[idx].item(), 4))
        for idx in top_indices
        if i2w[idx] != word
    ][:top_n]


print("\n--- [코사인 유사도 기반 상위 유사 단어 평가] ---")
test_words = ["영화", "최고의", "연기", "마블", "스토리"]
for tw in test_words:
    print(
        f"[{tw}] 와 유사한 단어 top 5:", get_similar_words(tw, test_model, loaded_meta)
    )

전체 샘플 수    : 58177
Train 샘플 수   : 46541 (80%)
Val 샘플 수     : 5818 (10%)
Test 샘플 수    : 5818 (10%)

--- CBOW 모델 학습 (Train & Validation) ---
Epoch 01/10 | Train Loss: 8.3439 | Val Loss: 8.0590 | Val Acc: 1.53%
Epoch 02/10 | Train Loss: 7.4689 | Val Loss: 7.7608 | Val Acc: 2.08%
Epoch 03/10 | Train Loss: 6.9977 | Val Loss: 7.7072 | Val Acc: 2.48%
Epoch 04/10 | Train Loss: 6.6157 | Val Loss: 7.6976 | Val Acc: 2.84%
Epoch 05/10 | Train Loss: 6.2503 | Val Loss: 7.7273 | Val Acc: 3.18%
Epoch 06/10 | Train Loss: 5.9039 | Val Loss: 7.7906 | Val Acc: 3.35%
Epoch 07/10 | Train Loss: 5.5789 | Val Loss: 7.8816 | Val Acc: 3.56%
Epoch 08/10 | Train Loss: 5.2783 | Val Loss: 7.9898 | Val Acc: 3.70%
Epoch 09/10 | Train Loss: 5.0023 | Val Loss: 8.1148 | Val Acc: 3.54%
Epoch 10/10 | Train Loss: 4.7498 | Val Loss: 8.2514 | Val Acc: 3.47%

[저장 완료] 'best_cbow_model.pth' 및 메타데이터 저장됨.

--- [Test Phase] 저장된 모델 로드 및 평가 ---
Test Dataset Loss : 7.6898
Test Accuracy     : 3.11% (181/5818 개 적중)

--- [코사인 유사도 기반 상위 유

#### 1-8 영화 리뷰 데이터셋(daum_movie_review.csv) + Skip-gram


In [ ]:
from transformers import AutoTokenizer
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import re
import pickle
from collections import Counter
from sklearn.model_selection import train_test_split

# -------------------------------------------------------------
# 1. 데이터 로드, 전처리 및 사전 구축
# -------------------------------------------------------------
df = pd.read_csv("../data/daum_movie_review.csv")
reviews = df["review"].dropna().tolist()


def clean_text(text):
    text = re.sub(r"[^가-힣a-zA-Z0-9\s]", "", str(text))
    return text.strip()


cleaned_reviews = [clean_text(r) for r in reviews if len(clean_text(r)) > 0]
tokens_list = [review.split() for review in cleaned_reviews]

# 단어 사전 구축 (최소 빈도수 5 이상)
MIN_COUNT = 5
word_counts = Counter([word for tokens in tokens_list for word in tokens])
vocab = [word for word, count in word_counts.items() if count >= MIN_COUNT]

word2idx = {word: i for i, word in enumerate(vocab)}
idx2word = {i: word for i, word in enumerate(vocab)}
vocab_size = len(vocab)

# -------------------------------------------------------------
# 2. Skip-gram (중심 단어 -> 주변 단어) 쌍 생성 및 데이터 분할
# -------------------------------------------------------------
WINDOW_SIZE = 2
skipgram_pairs = []

for tokens in tokens_list:
    indices = [word2idx[word] for word in tokens if word in word2idx]
    if len(indices) < WINDOW_SIZE * 2 + 1:
        continue
    for i in range(WINDOW_SIZE, len(indices) - WINDOW_SIZE):
        center_word = indices[i]
        context_words = (
            indices[i - WINDOW_SIZE : i] + indices[i + 1 : i + WINDOW_SIZE + 1]
        )
        for context_word in context_words:
            skipgram_pairs.append((center_word, context_word))

# Train(80%), Val(10%), Test(10%) 분할
train_pairs, temp_pairs = train_test_split(
    skipgram_pairs, test_size=0.2, random_state=42
)
val_pairs, test_pairs = train_test_split(temp_pairs, test_size=0.5, random_state=42)

print(f"전체 샘플 수    : {len(skipgram_pairs)}")
print(f"Train 샘플 수   : {len(train_pairs)} (80%)")
print(f"Val 샘플 수     : {len(val_pairs)} (10%)")
print(f"Test 샘플 수    : {len(test_pairs)} (10%)")


# Dataset 개선: __init__에서 미리 Tensor 변환
class SkipGramDataset(Dataset):
    def __init__(self, pairs):
        centers = [p[0] for p in pairs]
        contexts = [p[1] for p in pairs]
        self.centers = torch.tensor(centers, dtype=torch.long)
        self.contexts = torch.tensor(contexts, dtype=torch.long)

    def __len__(self):
        return len(self.centers)

    def __getitem__(self, idx):
        return self.centers[idx], self.contexts[idx]


train_loader = DataLoader(SkipGramDataset(train_pairs), batch_size=512, shuffle=True)
val_loader = DataLoader(SkipGramDataset(val_pairs), batch_size=512, shuffle=False)


# -------------------------------------------------------------
# 3. Skip-gram 모델 정의
# -------------------------------------------------------------
class SkipGramModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super(SkipGramModel, self).__init__()
        self.embeddings = nn.Embedding(vocab_size, embedding_dim)
        self.linear = nn.Linear(embedding_dim, vocab_size, bias=False)

    def forward(self, inputs):
        embeds = self.embeddings(inputs)
        out = self.linear(embeds)
        return out


EMBEDDING_DIM = 50
model = SkipGramModel(vocab_size, EMBEDDING_DIM)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.005)

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model.to(device)

# -------------------------------------------------------------
# 4. Train 및 Validation 학습 (Best Model 저장)
# -------------------------------------------------------------
epochs = 5
best_val_loss = float("inf")

print("\n--- Skip-gram 모델 학습 (Train & Validation) ---")
for epoch in range(epochs):
    # Train Phase
    model.train()
    train_loss = 0
    for center, context in train_loader:
        center, context = center.to(device), context.to(device)
        optimizer.zero_grad()
        output = model(center)
        loss = criterion(output, context)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    avg_train_loss = train_loss / len(train_loader)

    # Validation Phase
    model.eval()
    val_loss = 0
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for center, context in val_loader:
            center, context = center.to(device), context.to(device)
            output = model(center)
            loss = criterion(output, context)
            val_loss += loss.item()

            preds = torch.argmax(output, dim=1)
            val_correct += (preds == context).sum().item()
            val_total += context.size(0)

    avg_val_loss = val_loss / len(val_loader)
    val_acc = (val_correct / val_total) * 100

    print(
        f"Epoch {epoch + 1:02d}/{epochs} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.2f}%"
    )

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), "best_skipgram_model.pth")

test_meta = {
    "word2idx": word2idx,
    "idx2word": idx2word,
    "vocab_size": vocab_size,
    "embedding_dim": EMBEDDING_DIM,
    "test_pairs": test_pairs,
}
with open("skipgram_test_meta.pkl", "wb") as f:
    pickle.dump(test_meta, f)

print("\n[저장 완료] 'best_skipgram_model.pth' 및 메타데이터 저장됨.")

# -------------------------------------------------------------
# 5. 저장된 모델 로드 후 Test 데이터셋 평가
# -------------------------------------------------------------
print("\n--- [Test Phase] 저장된 모델 로드 및 평가 ---")

with open("skipgram_test_meta.pkl", "rb") as f:
    loaded_meta = pickle.load(f)

test_model = SkipGramModel(loaded_meta["vocab_size"], loaded_meta["embedding_dim"])
test_model.load_state_dict(torch.load("best_skipgram_model.pth"))
test_model.to(device)
test_model.eval()

loaded_test_loader = DataLoader(
    SkipGramDataset(loaded_meta["test_pairs"]), batch_size=512, shuffle=False
)

test_loss = 0
test_correct = 0
test_total = 0

with torch.no_grad():
    for center, context in loaded_test_loader:
        center, context = center.to(device), context.to(device)
        output = test_model(center)
        loss = criterion(output, context)
        test_loss += loss.item()

        preds = torch.argmax(output, dim=1)
        test_correct += (preds == context).sum().item()
        test_total += context.size(0)

avg_test_loss = test_loss / len(loaded_test_loader)
test_acc = (test_correct / test_total) * 100

print(f"Test Dataset Loss : {avg_test_loss:.4f}")
print(f"Test Accuracy     : {test_acc:.2f}% ({test_correct}/{test_total} 개 적중)")


# -------------------------------------------------------------
# 6. 저장된 모델 기준 코사인 유사도(Cosine Similarity) 평가
# -------------------------------------------------------------
def get_similar_words(word, model, meta, top_n=5):
    w2i = meta["word2idx"]
    i2w = meta["idx2word"]

    if word not in w2i:
        return f"'{word}' 단어가 어휘 사전에 없습니다."

    word_idx = torch.tensor([w2i[word]]).to(device)
    embed_weights = model.embeddings.weight.data

    target_vec = embed_weights[word_idx]

    norm_target = target_vec / torch.norm(target_vec, dim=1, keepdim=True)
    norm_weights = embed_weights / torch.norm(embed_weights, dim=1, keepdim=True)

    cosine_sim = torch.mm(norm_target, norm_weights.T).squeeze(0)
    top_indices = torch.topk(cosine_sim, top_n + 1).indices.tolist()

    return [
        (i2w[idx], round(cosine_sim[idx].item(), 4))
        for idx in top_indices
        if i2w[idx] != word
    ][:top_n]


print("\n--- [코사인 유사도 기반 상위 유사 단어 평가] ---")
test_words = ["영화", "최고의", "연기", "마블", "스토리"]
for tw in test_words:
    print(
        f"[{tw}] 와 유사한 단어 top 5:", get_similar_words(tw, test_model, loaded_meta)
    )

vocab_size :  4935
전체 샘플 수    : 232708
Train 샘플 수   : 186166 (80%)
Val 샘플 수     : 23271 (10%)
Test 샘플 수    : 23271 (10%)

--- Skip-gram 모델 학습 (Train & Validation) ---
Epoch 01/5 | Train Loss: 8.2439 | Val Loss: 7.8866 | Val Acc: 1.71%
Epoch 02/5 | Train Loss: 7.4652 | Val Loss: 7.7857 | Val Acc: 1.93%
Epoch 03/5 | Train Loss: 7.1834 | Val Loss: 7.8263 | Val Acc: 2.27%
Epoch 04/5 | Train Loss: 6.9962 | Val Loss: 7.8858 | Val Acc: 2.17%
Epoch 05/5 | Train Loss: 6.8412 | Val Loss: 7.9420 | Val Acc: 2.32%

[저장 완료] 'best_skipgram_model.pth' 및 메타데이터 저장됨.

--- [Test Phase] 저장된 모델 로드 및 평가 ---
Test Dataset Loss : 7.7760
Test Accuracy     : 2.02% (469/23271 개 적중)

--- [코사인 유사도 기반 상위 유사 단어 평가] ---
[영화] 와 유사한 단어 top 5: [('후반', 0.5538), ('춤', 0.5507), ('재미있네요', 0.5348), ('돈', 0.533), ('느낌은', 0.4883)]
[최고의] 와 유사한 단어 top 5: [('한심하다', 0.5008), ('뭔', 0.5007), ('있어서', 0.496), ('알려주는', 0.4878), ('결코', 0.4841)]
[연기] 와 유사한 단어 top 5: [('어색한', 0.5172), ('느낌이다', 0.5035), ('앉아서', 0.4878), ('싫다', 0.4754), ('처음이